# GeoAI Overview and Data Preparation

[GeoAI](https://github.com/opengeos/geoai) is a high-level Python toolkit for Earth observation machine learning. It wraps TorchGeo and segmentation_models.pytorch behind simpler APIs and adds:
- Direct integration with STAC catalogs and Earth Engine for data access
- Automatic chip generation from large GeoTIFFs
- Leafmap-based interactive visualization of results

**Architecture:** GeoAI sits on top of TorchGeo (data loading), segmentation_models.pytorch (architectures), and PyTorch Lightning (training). You interact with high-level `train_*()` functions rather than assembling these components manually.

## References
- GitHub: https://github.com/opengeos/geoai
- Book: https://book.opengeoai.org/
- Docs: https://geoai.gishub.org/

## 1. Verify Installation

In [ ]:
import geoai
print(f"GeoAI version: {geoai.__version__}")

# GeoAI is installed as 'geoai-py' on PyPI but imported as 'geoai'
print("\nKey sub-modules:")
print("  geoai.data       — chip generation, STAC access")
print("  geoai.train      — model training (segmentation, detection, change)")
print("  geoai.inference  — tiled inference on large rasters")
print("  geoai.visualize  — Leafmap integration for interactive maps")

## 2. Download Sample Training Data

GeoAI includes utilities to download sample datasets for quick experimentation.

In [ ]:
import os
from geoai import data as geoai_data

# Download a sample building segmentation dataset
dataset_dir = "sample_data"
os.makedirs(dataset_dir, exist_ok=True)

# GeoAI provides sample imagery + label pairs for quick starts
image_url, label_url = geoai_data.get_building_sample_urls()

import urllib.request
image_path = os.path.join(dataset_dir, "image.tif")
label_path = os.path.join(dataset_dir, "labels.tif")

if not os.path.exists(image_path):
    urllib.request.urlretrieve(image_url, image_path)
if not os.path.exists(label_path):
    urllib.request.urlretrieve(label_url, label_path)

print(f"Downloaded image to {image_path}")
print(f"Downloaded labels to {label_path}")

## 3. Explore the Data

In [ ]:
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

with rasterio.open(image_path) as src:
    show(src, ax=axes[0], title="RGB Image")
    print(f"Image: {src.width}x{src.height}px, {src.count} bands, CRS={src.crs}")

with rasterio.open(label_path) as src:
    labels = src.read(1)
    axes[1].imshow(labels, cmap="tab10")
    axes[1].set_title(f"Labels (classes: {np.unique(labels)})")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4. Generate Training Chips

Large rasters must be chipped into small patches for training. `geoai.chip_raster()` tiles the image and label rasters together, ensuring chips are aligned.

In [ ]:
from geoai import chip_raster

chip_dir = "chips"
os.makedirs(chip_dir, exist_ok=True)

chip_raster(
    image=image_path,
    labels=label_path,
    output_dir=chip_dir,
    chip_size=256,
    stride=128,          # 50% overlap between adjacent chips
    drop_empty=True,     # discard chips with no labeled pixels
)

image_chips = [f for f in os.listdir(os.path.join(chip_dir, "images")) if f.endswith(".tif")]
label_chips = [f for f in os.listdir(os.path.join(chip_dir, "labels")) if f.endswith(".tif")]
print(f"Generated {len(image_chips)} image chips and {len(label_chips)} label chips")

## 5. Visualize Sample Chips

In [ ]:
import random

sample_chips = random.sample(image_chips, min(4, len(image_chips)))

fig, axes = plt.subplots(2, len(sample_chips), figsize=(4 * len(sample_chips), 8))

for i, chip_name in enumerate(sample_chips):
    img_chip_path = os.path.join(chip_dir, "images", chip_name)
    lbl_chip_path = os.path.join(chip_dir, "labels", chip_name)

    with rasterio.open(img_chip_path) as src:
        show(src, ax=axes[0, i])
        axes[0, i].set_title(f"Image chip {i+1}")

    if os.path.exists(lbl_chip_path):
        with rasterio.open(lbl_chip_path) as src:
            axes[1, i].imshow(src.read(1), cmap="tab10")
            axes[1, i].set_title(f"Label chip {i+1}")
            axes[1, i].axis("off")

plt.suptitle("Sample training chips (image + label pairs)")
plt.tight_layout()
plt.show()

## 6. GeoAI Workflow Summary

```
Raw large GeoTIFF
      │
      ▼  geoai.chip_raster()
Training chips (256×256 px image+label pairs)
      │
      ▼  geoai.train_segmentation()  /  .train_detection()  /  .train_change_detection()
Trained model
      │
      ▼  geoai.predict()  (tiled inference)
Full-scene prediction GeoTIFF
      │
      ▼  geoai.Map()  (Leafmap interactive visualization)
Interactive web map
```

Each step is a single function call. Continue to `1-segmentation_training.ipynb` to train a model.